In [ ]:
# ============================================================
# HMM POS TAGGER USING UNIVERSAL DEPENDENCIES ENGLISH EWT
# Dataset is automatically downloaded by this code
# ============================================================

import urllib.request
from collections import defaultdict, Counter
import math
import os

# ------------------------------------------------------------
# 1. AUTOMATICALLY DOWNLOAD DATASET
# ------------------------------------------------------------

print("Downloading Universal Dependencies English EWT dataset...")

BASE_URL = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/master/"

train_url = BASE_URL + "en_ewt-ud-train.conllu"
test_url = BASE_URL + "en_ewt-ud-test.conllu"

urllib.request.urlretrieve(train_url, "train.conllu")
urllib.request.urlretrieve(test_url, "test.conllu")

print("Dataset downloaded successfully!")
print("Training file: train.conllu")
print("Testing file : test.conllu")


# ------------------------------------------------------------
# 2. READ CONLLU DATASET
# ------------------------------------------------------------

def read_conllu(filename):

    sentences = []

    words = []
    tags = []

    with open(filename, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            # End of sentence
            if line == "":
                if words:
                    sentences.append((words, tags))
                    words = []
                    tags = []
                continue

            # Ignore comments
            if line.startswith("#"):
                continue

            columns = line.split("\t")

            # CoNLL-U has 10 columns
            if len(columns) != 10:
                continue

            word_id = columns[0]

            # Ignore multi-word tokens and empty nodes
            if "-" in word_id or "." in word_id:
                continue

            word = columns[1]
            pos = columns[3]

            words.append(word)
            tags.append(pos)

    return sentences


# Read training and testing data

train_data = read_conllu("train.conllu")
test_data = read_conllu("test.conllu")

print("\nTraining sentences:", len(train_data))
print("Testing sentences :", len(test_data))


# ------------------------------------------------------------
# 3. CALCULATE TRANSITION AND EMISSION COUNTS
# ------------------------------------------------------------

transition_counts = defaultdict(Counter)
emission_counts = defaultdict(Counter)
tag_counts = Counter()

START = "<START>"
END = "<END>"

for words, tags in train_data:

    previous_tag = START

    for word, tag in zip(words, tags):

        # Transition probability count
        transition_counts[previous_tag][tag] += 1

        # Emission probability count
        emission_counts[tag][word.lower()] += 1

        # POS tag count
        tag_counts[tag] += 1

        previous_tag = tag

    # Last tag -> END
    transition_counts[previous_tag][END] += 1


tags_list = list(tag_counts.keys())

print("\nNumber of POS tags:", len(tags_list))
print("POS Tags:", tags_list)


# ------------------------------------------------------------
# 4. CALCULATE TRANSITION PROBABILITIES
# ------------------------------------------------------------

transition_prob = defaultdict(dict)

for previous_tag in transition_counts:

    total = sum(transition_counts[previous_tag].values())

    for current_tag in transition_counts[previous_tag]:

        transition_prob[previous_tag][current_tag] = (
            transition_counts[previous_tag][current_tag] / total
        )


# ------------------------------------------------------------
# 5. CALCULATE EMISSION PROBABILITIES
# ------------------------------------------------------------

emission_prob = defaultdict(dict)

for tag in emission_counts:

    total = sum(emission_counts[tag].values())

    for word in emission_counts[tag]:

        emission_prob[tag][word] = (
            emission_counts[tag][word] / total
        )


# ------------------------------------------------------------
# 6. VITERBI ALGORITHM
# ------------------------------------------------------------

def viterbi(sentence):

    words = sentence.split()

    viterbi_table = []
    backpointer = []

    # --------------------------------------------------------
    # FIRST WORD
    # --------------------------------------------------------

    first_word = words[0].lower()

    first_scores = {}
    first_backpointer = {}

    for tag in tags_list:

        transition = transition_prob[START].get(tag, 1e-6)

        emission = emission_prob[tag].get(first_word, 1e-6)

        first_scores[tag] = (
            math.log(transition) +
            math.log(emission)
        )

        first_backpointer[tag] = START

    viterbi_table.append(first_scores)
    backpointer.append(first_backpointer)


    # --------------------------------------------------------
    # REMAINING WORDS
    # --------------------------------------------------------

    for i in range(1, len(words)):

        word = words[i].lower()

        current_scores = {}
        current_backpointer = {}

        for current_tag in tags_list:

            emission = emission_prob[current_tag].get(
                word, 1e-6
            )

            best_score = float("-inf")
            best_previous_tag = None

            for previous_tag in tags_list:

                transition = transition_prob[
                    previous_tag
                ].get(current_tag, 1e-6)

                score = (
                    viterbi_table[i - 1][previous_tag]
                    + math.log(transition)
                    + math.log(emission)
                )

                if score > best_score:

                    best_score = score
                    best_previous_tag = previous_tag

            current_scores[current_tag] = best_score
            current_backpointer[current_tag] = best_previous_tag

        viterbi_table.append(current_scores)
        backpointer.append(current_backpointer)


    # --------------------------------------------------------
    # FIND BEST FINAL TAG
    # --------------------------------------------------------

    best_final_score = float("-inf")
    best_final_tag = None

    for tag in tags_list:

        transition = transition_prob[tag].get(END, 1e-6)

        score = (
            viterbi_table[-1][tag]
            + math.log(transition)
        )

        if score > best_final_score:

            best_final_score = score
            best_final_tag = tag


    # --------------------------------------------------------
    # BACKTRACK
    # --------------------------------------------------------

    best_tags = [best_final_tag]

    for i in range(len(words) - 1, 0, -1):

        previous_tag = backpointer[i][best_tags[-1]]

        best_tags.append(previous_tag)

    best_tags.reverse()

    return list(zip(words, best_tags))


# ------------------------------------------------------------
# 7. ENTER SENTENCE
# ------------------------------------------------------------

sentence = input(
    "\nEnter a sentence: "
)

result = viterbi(sentence)


# ------------------------------------------------------------
# 8. DISPLAY POS TAGGING RESULT
# ------------------------------------------------------------

print("\n======================================")
print("HMM POS TAGGING RESULT")
print("======================================")

for word, tag in result:

    print(f"{word:15} -> {tag}")


# ------------------------------------------------------------
# 9. ACCURACY ON TEST DATASET
# ------------------------------------------------------------

print("\nCalculating accuracy on test dataset...")

correct = 0
total = 0

# Test first 100 sentences to keep execution faster
for words, actual_tags in test_data[:100]:

    sentence = " ".join(words)

    predicted_result = viterbi(sentence)

    predicted_tags = [
        tag for word, tag in predicted_result
    ]

    for predicted, actual in zip(
        predicted_tags,
        actual_tags
    ):

        if predicted == actual:
            correct += 1

        total += 1


accuracy = (correct / total) * 100

print("\n======================================")
print("POS TAGGING ACCURACY")
print("======================================")

print("Correct tags :", correct)
print("Total tags   :", total)
print("Accuracy     :", round(accuracy, 2), "%")

Dataset downloaded successfully!
Training file: train.conllu
Testing file : test.conllu

Training sentences: 12544
Testing sentences : 2077

Number of POS tags: 17
POS Tags: ['PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'DET', 'ADP', 'AUX', 'PRON', 'PART', 'SCONJ', 'NUM', 'ADV', 'CCONJ', 'INTJ', 'X', 'SYM']
